## In this notebook, we use different ensemble techniques to improve model performance. We start building models after hyperparam tuning, using methods like Voting, Bagging, and Stacking to boost accuracy and generalization. We evaluate performance with metrics such as accuracy, bias, and variance.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingClassifier, VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsOneClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('winequality-red-clean.csv', delimiter=',')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1458 non-null   float64
 1   volatile acidity      1458 non-null   float64
 2   citric acid           1458 non-null   float64
 3   residual sugar        1458 non-null   float64
 4   chlorides             1458 non-null   float64
 5   free sulfur dioxide   1458 non-null   float64
 6   total sulfur dioxide  1458 non-null   float64
 7   density               1458 non-null   float64
 8   pH                    1458 non-null   float64
 9   sulphates             1458 non-null   float64
 10  alcohol               1458 non-null   float64
 11  quality               1458 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 136.8 KB


In [2]:
'''
Selected Features classification:
 Index(['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
       'alcohol'], dtype='object')
'''

selected_features = ['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
                     'alcohol']
X_selected = df[selected_features]
y = df['quality']

standard_scaler = StandardScaler()
X_selected = standard_scaler.fit_transform(X_selected)

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)


In [3]:
def train_test_accuracy(train_accuracy, test_accuracy):
    print(f'Training Accuracy: {(train_accuracy * 100).__round__(3)}%')
    print(f'Testing Accuracy: {(test_accuracy * 100).__round__(3)}%')

In [4]:
def calculate_bias_variance(model, X_test, y_test):
    predictions = model.predict(X_test)

    mean_prediction = np.mean(predictions)
    true_mean = np.mean(y_test)
    bias = mean_prediction - true_mean

    variance = np.mean((predictions - mean_prediction) ** 2)

    print(f"Bias: {bias}")
    print(f"Variance: {variance}")

    bias_threshold = 0.15
    variance_threshold = 0.2

    if abs(bias) > bias_threshold and variance < variance_threshold:
        print("Model Status: Underfitting - High bias and low variance indicate the model is too simplistic.")
    elif abs(bias) < bias_threshold and variance > variance_threshold:
        print("Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.")
    elif abs(bias) > bias_threshold and variance > variance_threshold:
        print(
            "Model Status: Unbalanced - Both bias and variance are high, suggesting the model may not generalize well.")
    else:
        print("Model Status: Balanced - The model has a good balance between bias and variance.")

 ### We set up multiple machine learning models using the optimal parameters obtained from tuning. These include KNN, SVM (OneVsRest and OneVsOne with linear kernels), and 2 Extra Trees models. The selected parameters are intended to enhance the accuracy of each model, and the combined use of these models in an ensemble aims to improve overall predictive performance.

In [5]:
from sklearn.ensemble import ExtraTreesClassifier

# KNN, SVM and RandomForest Models
knn_1 = KNeighborsClassifier(weights='distance', metric='manhattan', algorithm='ball_tree', n_neighbors=18)

ovr_linear_model = OneVsRestClassifier(
    SVC(kernel='linear', C=1000, probability=True))  # Best Parameters: {'estimator__C': 1000} # First best

ovo_linear_model = OneVsOneClassifier(
    SVC(kernel='linear', C=0.1, probability=True))  #Best Parameters: {'estimator__C': 10} # Second Best

et = ExtraTreesClassifier(random_state=42)

et_boot = ExtraTreesClassifier(bootstrap= True, max_depth= 20, max_features='sqrt', min_samples_leaf= 1, min_samples_split= 2, n_estimators= 200)  

## ***Hard voting***: Selecting the class that gets the most votes from all classifiers, without using predicted probabilities.
### In this cell, we construct an ensemble model using a VotingClassifier with hard voting, which includes KNN, OneVsRest SVM, OneVsOne SVM, and two Extra Trees models


In [6]:
#Voting(Hard)
voting_clf_hard = VotingClassifier(estimators=[
    ('knn_1', knn_1),
    ('ovr_linear_model', ovr_linear_model),
    ('ovo_linear_model', ovo_linear_model),
    ('rcf_1', et),
    ('rcf_2', et_boot)
], voting='hard')

voting_clf_hard.fit(X_train, y_train)

y_pred = voting_clf_hard.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Ensemble Voting(Hard) Classifier Accuracy: {accuracy * 100:.2f}%")
calculate_bias_variance(voting_clf_hard, X_test, y_test)

Ensemble Voting(Hard) Classifier Accuracy: 70.21%
Bias: -0.0582191780821919
Variance: 0.3794684743854382
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


## ***Soft voting***: Averaging the predicted probabilities of each class across all classifiers, and selecting the class with the highest average probability.

### In this cell, we construct an ensemble model using a VotingClassifier with soft voting, including KNN, OneVsRest SVM, and two Extra trees models. The OneVsOne SVM was excluded due to its lack of the predict_proba method, which makes it incompatible with soft voting.


In [7]:
# Voting(Soft)
voting_clf_soft = VotingClassifier(estimators=[
    ('knn_1', knn_1),
    ('ovr_linear_model', ovr_linear_model),
    # ('ovo_linear_model', ovo_linear_model), # 'OneVsOneClassifier' object has no attribute 'predict_proba' because it does not inherently aggregate probabilities across binary classifiers like OneVsRestClassifier does.
    ('rcf_1', et),
    ('rcf_2', et_boot)
], voting='soft')

voting_clf_soft.fit(X_train, y_train)

y_pred = voting_clf_soft.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Ensemble Voting(Soft) Classifier Accuracy: {accuracy * 100:.2f}%")
calculate_bias_variance(voting_clf_soft, X_test, y_test)

Ensemble Voting(Soft) Classifier Accuracy: 69.18%
Bias: -0.04794520547945247
Variance: 0.3563520360292738
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


## ***Bagging (Bootstrap Aggregating)***: Is an ensemble method where multiple versions of a base model are trained on different random samples, and their predictions are combined to improve stability and accuracy.
### In this cell, we create bagged versions of KNN, OneVsRest SVM, OneVsOne SVM, and Extra trees using BaggingClassifier, with 20 estimators for most models to increase diversity. A VotingClassifier is then constructed using these bagged models with hard voting.

In [8]:
# Bagging with Voting(hard)

bagged_knn_1 = BaggingClassifier(knn_1, n_estimators=20, random_state=42, n_jobs=-1)
bagged_ovr_linear = BaggingClassifier(ovr_linear_model, n_estimators=20, random_state=42, n_jobs=-1)
bagged_ovo_linear = BaggingClassifier(ovo_linear_model, n_estimators=20, random_state=42, n_jobs=-1)
bagging_rfc_1 = BaggingClassifier(et, n_estimators=1, random_state=42)
bagging_rfc_2 = BaggingClassifier(et_boot, n_estimators=1, random_state=42)

voting_hard_clf_bagged = VotingClassifier(estimators=[
    ('bagged_knn_1', bagged_knn_1),
    ('bagging_rfc_1', bagging_rfc_1),
    ('bagging_rfc_2', bagging_rfc_2),
    ('bagged_ovr_linear', bagged_ovr_linear),
    ('bagged_ovo_linear', bagged_ovo_linear),
], voting='hard', n_jobs=-1)

voting_hard_clf_bagged.fit(X_train, y_train)

y_pred_bagged = voting_hard_clf_bagged.predict(X_test)
accuracy_bagged = accuracy_score(y_test, y_pred_bagged)
print(f"Accuracy (Bagged Voting(Hard) Classifier with 3 estimators): {accuracy_bagged * 100:.2f}%")
calculate_bias_variance(voting_hard_clf_bagged, X_test, y_test)

Accuracy (Bagged Voting(Hard) Classifier with 3 estimators): 67.81%
Bias: -0.04452054794520599
Variance: 0.3622982735972978
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


### In this cell, we construct an ensemble using bagged versions of several models, including KNN, OneVsRest SVM, and two Extra trees models. Soft voting is applied to average the predicted probabilities and select the most likely class. We excluded the bagged OneVsOne SVM due to its incompatibility with probability predictions. The ensemble is trained and evaluated on the test set for accuracy, bias, and variance.

In [9]:
# Bagging with Voting(soft)
bagged_knn_1 = BaggingClassifier(knn_1, n_estimators=20, random_state=42, n_jobs=-1)
bagged_ovr_linear = BaggingClassifier(ovr_linear_model, n_estimators=20, random_state=42, n_jobs=-1)
# bagged_ovo_linear = BaggingClassifier(ovo_linear_model, n_estimators=20, random_state=42, n_jobs=-1)
bagging_rfc_1 = BaggingClassifier(et, n_estimators=1, random_state=42)
bagging_rfc_2 = BaggingClassifier(et_boot, n_estimators=1, random_state=42)

voting_soft_clf_bagged = VotingClassifier(estimators=[
    ('bagged_knn_1', bagged_knn_1),
    ('bagging_rfc_1', bagging_rfc_1),
    ('bagging_rfc_2', bagging_rfc_2),
    ('bagged_ovr_linear', bagged_ovr_linear),
    # ('bagged_ovo_linear', bagged_ovo_linear),
], voting='soft', n_jobs=-1)

voting_soft_clf_bagged.fit(X_train, y_train)

y_pred_bagged = voting_soft_clf_bagged.predict(X_test)
accuracy_bagged = accuracy_score(y_test, y_pred_bagged)
print(f"Accuracy (Bagged Voting(Hard) Classifier with 3 estimators): {accuracy_bagged * 100:.2f}%")
calculate_bias_variance(voting_soft_clf_bagged, X_test, y_test)

Accuracy (Bagged Voting(Hard) Classifier with 3 estimators): 67.81%
Bias: -0.04452054794520599
Variance: 0.3622982735972978
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


##  ***Stacking***: Is an ensemble learning technique where multiple base models' predictions are combined using a final estimator, often called a metamodel, to make the final prediction.
### In this cell, we construct a stacking ensemble with several base models, including KNN, OneVsRest SVM, OneVsOne SVM, and Extra Trees, with a Logistic Regression metamodel.

In [10]:
#Stacking
stacking_ensemble = StackingClassifier(
    estimators=[
        ('knn_base', knn_1),
        ('ovr_linear', ovr_linear_model),
        ('ovo_linear', ovo_linear_model),
        ('rfc_1', et),
        ('rfc_2', et_boot)
    ],
    final_estimator=LogisticRegression(),  # Metamodel to make the final prediction
    cv=5
)

stacking_ensemble.fit(X_train, y_train)

test_accuracy = stacking_ensemble.score(X_test, y_test)
print(f"Test accuracy of the stacking ensemble: {test_accuracy:.4f}")
calculate_bias_variance(stacking_ensemble, X_test, y_test)

Test accuracy of the stacking ensemble: 0.6986
Bias: -0.023972602739726234
Variance: 0.37008585100394065
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


C:\Users\margi\PycharmProjects\data_5\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### All ensemble methods showed consistent overfitting, with low bias and high variance. Hard voting achieved the highest accuracy (70.21%), closely followed by soft voting (69.18%), but both overfit. Bagging performed slightly worse, with a maximum accuracy of 68.84%. The stacking ensemble matched the accuracy of hard voting at 70.21% but still overfitted. Overall, stacking and hard voting performed best, with hard voting being recommended due to its simplicity and similar performance.